In [1]:
# ── Cell R1: install (same as always — no torch) ──
!pip install -q transformers datasets accelerate peft bitsandbytes huggingface_hub
!pip uninstall -y torchao
print("✅ RESTART: Runtime > Restart session, then run Cell R2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.9 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
✅ RESTART: Runtime > Restart session, then run Cell R2.


In [1]:
# ── Cell R2: mount Drive + restore both adapters from your backup ──
from google.colab import drive
drive.mount("/content/drive")
!cp -r "/content/drive/MyDrive/gpt2-medqa-lora"        gpt2-medqa-lora
!cp -r "/content/drive/MyDrive/tinyllama-medqa-qlora"  tinyllama-medqa-qlora
!ls -la gpt2-medqa-lora tinyllama-medqa-qlora   # confirm adapter files are here

Mounted at /content/drive
gpt2-medqa-lora:
total 9860
drwx------ 3 root root    4096 Aug  3 13:25 .
drwxr-xr-x 1 root root    4096 Aug  3 13:25 ..
-rw------- 1 root root    1005 Aug  3 13:25 adapter_config.json
-rw------- 1 root root 6497232 Aug  3 13:25 adapter_model.safetensors
drwx------ 2 root root    4096 Aug  3 13:25 checkpoint-923
-rw------- 1 root root    5162 Aug  3 13:25 README.md
-rw------- 1 root root     326 Aug  3 13:25 tokenizer_config.json
-rw------- 1 root root 3557680 Aug  3 13:25 tokenizer.json
-rw------- 1 root root    5201 Aug  3 13:25 training_args.bin

tinyllama-medqa-qlora:
total 12396
drwx------ 3 root root    4096 Aug  3 13:25 .
drwxr-xr-x 1 root root    4096 Aug  3 13:25 ..
-rw------- 1 root root    1064 Aug  3 13:25 adapter_config.json
-rw------- 1 root root 9034656 Aug  3 13:25 adapter_model.safetensors
-rw------- 1 root root     410 Aug  3 13:25 chat_template.jinja
drwx------ 2 root root    4096 Aug  3 13:25 checkpoint-923
-rw------- 1 root root    1536 Au

In [2]:
# ── Cell 23: authenticate to the Hub with a WRITE token ──
# Create one at huggingface.co/settings/tokens (role: Write), then paste when prompted.
from huggingface_hub import notebook_login
notebook_login()

In [3]:
# ── Cell 24: push both adapters as their own model repos ──
from huggingface_hub import whoami
user = whoami()["name"]
print("pushing as:", user)

# push_to_hub creates the repo and uploads adapter weights + config + tokenizer.
gpt2_chat.model.push_to_hub(f"{user}/gpt2-medqa-lora")
gpt2_chat.tokenizer.push_to_hub(f"{user}/gpt2-medqa-lora")

tinyllama_chat.model.push_to_hub(f"{user}/tinyllama-medqa-qlora")
tinyllama_chat.tokenizer.push_to_hub(f"{user}/tinyllama-medqa-qlora")

print(f"✅ pushed:\n  {user}/gpt2-medqa-lora\n  {user}/tinyllama-medqa-qlora")

pushing as: Babblu2821


NameError: name 'gpt2_chat' is not defined

In [4]:
# ── verify the restored folders exist ──
!ls -la gpt2-medqa-lora tinyllama-medqa-qlora

gpt2-medqa-lora:
total 9860
drwx------ 3 root root    4096 Aug  3 13:25 .
drwxr-xr-x 1 root root    4096 Aug  3 13:25 ..
-rw------- 1 root root    1005 Aug  3 13:25 adapter_config.json
-rw------- 1 root root 6497232 Aug  3 13:25 adapter_model.safetensors
drwx------ 2 root root    4096 Aug  3 13:25 checkpoint-923
-rw------- 1 root root    5162 Aug  3 13:25 README.md
-rw------- 1 root root     326 Aug  3 13:25 tokenizer_config.json
-rw------- 1 root root 3557680 Aug  3 13:25 tokenizer.json
-rw------- 1 root root    5201 Aug  3 13:25 training_args.bin

tinyllama-medqa-qlora:
total 12396
drwx------ 3 root root    4096 Aug  3 13:25 .
drwxr-xr-x 1 root root    4096 Aug  3 13:25 ..
-rw------- 1 root root    1064 Aug  3 13:25 adapter_config.json
-rw------- 1 root root 9034656 Aug  3 13:25 adapter_model.safetensors
-rw------- 1 root root     410 Aug  3 13:25 chat_template.jinja
drwx------ 2 root root    4096 Aug  3 13:25 checkpoint-923
-rw------- 1 root root    1536 Aug  3 13:25 README.md
-rw--

In [5]:
# ── Cell 24 (FIXED): push adapter FOLDERS straight to the Hub ──
from huggingface_hub import HfApi, whoami
api  = HfApi()
user = whoami()["name"]          # Babblu2821
print("pushing as:", user)

for folder, repo in [("gpt2-medqa-lora",       f"{user}/gpt2-medqa-lora"),
                     ("tinyllama-medqa-qlora", f"{user}/tinyllama-medqa-qlora")]:
    api.create_repo(repo, repo_type="model", exist_ok=True)   # make repo if absent
    api.upload_folder(folder_path=folder, repo_id=repo, repo_type="model")
    print(f"✅ uploaded {folder}  ->  {repo}")

pushing as: Babblu2821
✅ uploaded gpt2-medqa-lora  ->  Babblu2821/gpt2-medqa-lora
✅ uploaded tinyllama-medqa-qlora  ->  Babblu2821/tinyllama-medqa-qlora
